In [0]:
# Databricks notebook source
# ============================================================
# NOTEBOOK: nb_05_PortfolioExceptionReport
# PURPOSE:  Replaces SQL Procedure 5 (usp_LoadPortfolioExceptionReport)
#           Builds the daily operational Portfolio Exception Report
#           used by Business, Operations, Risk, Finance and Data
#           Governance teams.
#
#           The report consolidates:
#             - Executive KPIs (portfolio health, exception rate)
#             - Source system quality metrics
#             - Business area KPIs
#             - Business owner workload
#             - SLA performance
#             - 30-day trend analysis
#             - Executive scorecard (GREEN/AMBER/RED)
#
# CATALOG:  retailbank_dev
# ============================================================

import uuid
from datetime import datetime
from delta.tables import DeltaTable
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# --------------------------------------------------------
# PARAMETERS
# --------------------------------------------------------
dbutils.widgets.text("business_date", "2026-01-31", "Business Date")
dbutils.widgets.text("debug", "1", "Debug Mode (1=print, 0=silent)")

business_date = dbutils.widgets.get("business_date")
debug         = int(dbutils.widgets.get("debug"))

# --------------------------------------------------------
# AUDIT VARIABLES
# --------------------------------------------------------
execution_id   = str(uuid.uuid4())
procedure_name = "nb_05_PortfolioExceptionReport"
start_time     = datetime.now()

rows_inserted = 0

if debug:
    print("=" * 50)
    print("NB_05_PORTFOLIOEXCEPTIONREPORT STARTED")
    print("=" * 50)
    print(f"Execution ID : {execution_id}")
    print(f"Business Date: {business_date}")

NB_05_PORTFOLIOEXCEPTIONREPORT STARTED
Execution ID : 5ed9695c-a6f9-432a-b928-0eb370f01ea4
Business Date: 2026-01-31


In [0]:
# Databricks notebook source
# ============================================================
# AUDIT LOG FUNCTIONS
# Same pattern as nb_01, nb_02, nb_03, and nb_04.
# ============================================================

def write_audit_start():
    sql = f"""
        INSERT INTO retailbank_dev.audit.etl_execution_log 
        (execution_id, procedure_name, business_date, start_time, status)
        VALUES 
        ('{execution_id}', '{procedure_name}', '{business_date}',
         '{start_time.strftime("%Y-%m-%d %H:%M:%S")}', 'RUNNING')
    """
    spark.sql(sql)


def write_audit_end(status, message):
    end_time         = datetime.now()
    duration_seconds = int((end_time - start_time).total_seconds())
    safe_message     = message.replace("'", "''")
    
    sql = f"""
        UPDATE retailbank_dev.audit.etl_execution_log
        SET 
            end_time         = '{end_time.strftime("%Y-%m-%d %H:%M:%S")}',
            status           = '{status}',
            rows_inserted    = {rows_inserted},
            duration_seconds = {duration_seconds},
            message          = '{safe_message}'
        WHERE execution_id = '{execution_id}'
    """
    spark.sql(sql)


write_audit_start()

In [0]:
# Databricks notebook source
# ============================================================
# READ SOURCE DATA
# 1. Portfolio Exceptions -> what we report on
# 2. Enterprise Portfolio -> account counts and source metrics
# ============================================================

# 3.1 Read Portfolio Exceptions for the business date
exceptions_df = spark.table("retailbank_dev.warehouse.customer_portfolio_exceptions") \
    .filter(F.col("business_date") == business_date)

# 3.2 Read Enterprise Portfolio for the business date
portfolio_df = spark.table("retailbank_dev.warehouse.customer_portfolio") \
    .filter(F.col("business_date") == business_date)

if debug:
    print("Source data loaded:")
    print(f"  Exceptions found: {exceptions_df.count()}")
    print(f"  Portfolio records: {portfolio_df.count()}")

Source data loaded:
  Exceptions found: 6
  Portfolio records: 19


In [0]:
# Databricks notebook source
# ============================================================
# CALCULATE EXECUTIVE KPIs
# 
# These are the high-level metrics that appear on the
# executive dashboard:
#   - TotalAccounts     = All accounts in the portfolio
#   - TotalCustomers    = Unique customers in the portfolio
#   - TotalExceptions   = All exceptions for the business date
#   - ExceptionRate     = (TotalExceptions / TotalAccounts) * 100
#   - CriticalExceptions, High, Medium, Low = Counts by severity
# ============================================================

# Calculate portfolio totals
portfolio_totals = portfolio_df.agg(
    F.count("*").alias("total_accounts"),
    F.countDistinct("customer_id").alias("total_customers")
).collect()[0]

total_accounts   = portfolio_totals.total_accounts or 0
total_customers  = portfolio_totals.total_customers or 0

# Calculate exception totals by severity
if exceptions_df.count() > 0:
    exception_totals = exceptions_df.groupBy().agg(
        F.count("*").alias("total_exceptions"),
        F.sum(F.when(F.col("severity_code") == "CRITICAL", 1).otherwise(0)).alias("critical_exceptions"),
        F.sum(F.when(F.col("severity_code") == "HIGH", 1).otherwise(0)).alias("high_exceptions"),
        F.sum(F.when(F.col("severity_code") == "MEDIUM", 1).otherwise(0)).alias("medium_exceptions"),
        F.sum(F.when(F.col("severity_code") == "LOW", 1).otherwise(0)).alias("low_exceptions")
    ).collect()[0]
    
    total_exceptions    = exception_totals.total_exceptions or 0
    critical_exceptions = exception_totals.critical_exceptions or 0
    high_exceptions     = exception_totals.high_exceptions or 0
    medium_exceptions   = exception_totals.medium_exceptions or 0
    low_exceptions      = exception_totals.low_exceptions or 0
else:
    total_exceptions = 0
    critical_exceptions = 0
    high_exceptions = 0
    medium_exceptions = 0
    low_exceptions = 0

# Calculate Exception Rate (percentage)
if total_accounts > 0:
    exception_rate = round((total_exceptions / total_accounts) * 100, 2)
else:
    exception_rate = 0.0

if debug:
    print("Executive KPIs:")
    print(f"  Total Customers  : {total_customers}")
    print(f"  Total Accounts   : {total_accounts}")
    print(f"  Total Exceptions : {total_exceptions}")
    print(f"  Exception Rate   : {exception_rate}%")
    print(f"  Critical         : {critical_exceptions}")
    print(f"  High             : {high_exceptions}")
    print(f"  Medium           : {medium_exceptions}")
    print(f"  Low              : {low_exceptions}")

Executive KPIs:
  Total Customers  : 12
  Total Accounts   : 19
  Total Exceptions : 6
  Exception Rate   : 31.58%
  Critical         : 3
  High             : 3
  Medium           : 0
  Low              : 0


In [0]:
# Databricks notebook source
# ============================================================
# CALCULATE PORTFOLIO HEALTH
# 
# Portfolio Health is based on the Exception Rate.
#   EXCELLENT: < 1%
#   GOOD:      1% - 3%
#   FAIR:      3% - 5%
#   POOR:      > 5%
#
# Executive Status is based on critical and high exceptions.
#   Immediate Attention: > 50 Critical exceptions
#   Management Review:   > 200 High exceptions
#   Normal Operations:   Otherwise
# ============================================================

# Determine Portfolio Health
if exception_rate < 1:
    portfolio_health = "EXCELLENT"
elif exception_rate < 3:
    portfolio_health = "GOOD"
elif exception_rate < 5:
    portfolio_health = "FAIR"
else:
    portfolio_health = "POOR"

# Determine Executive Status
if critical_exceptions > 50:
    executive_status = "Immediate Attention"
elif high_exceptions > 200:
    executive_status = "Management Review"
else:
    executive_status = "Normal Operations"

if debug:
    print("Portfolio Health:")
    print(f"  Health Score    : {portfolio_health}")
    print(f"  Executive Status: {executive_status}")

Portfolio Health:
  Health Score    : POOR
  Executive Status: Normal Operations


In [0]:
# ============================================================
# SOURCE SYSTEM QUALITY METRICS
# ============================================================
#
# WHAT THIS CELL DOES:
# For each of the seven source systems (CORE_BANKING, LOANS,
# CARDS, INVESTMENTS, MOBILE, MORTGAGE, FOREX) it calculates:
#   - total_accounts     : how many accounts came from this source
#   - total_exceptions   : how many exceptions involve this source
#   - critical_exceptions: how many are CRITICAL severity
#   - high_exceptions    : how many are HIGH severity
#   - exception_rate     : exceptions / accounts * 100
#
# This helps identify which source systems have the most data
# quality issues and where remediation effort should focus.
#
# BUG FIXED IN THIS VERSION:
# --------------------------------------------------------
# The previous version created an empty fallback DataFrame
# using source_accounts.schema when no exceptions existed:
#
#   source_exceptions = spark.createDataFrame([], source_accounts.schema)
#
# source_accounts.schema only has TWO columns:
#   source_system_code, total_accounts
#
# But the JOIN below then referenced FOUR columns from
# source_exceptions:
#   e.total_exceptions, e.critical_exceptions,
#   e.high_exceptions, e.source_system_code
#
# When exceptions_df was empty, those columns did not exist
# in the fallback DataFrame, causing:
#   "column e.total_exceptions cannot be resolved"
#
# THE FIX:
# Create the empty fallback DataFrame with the CORRECT schema
# that matches what the JOIN and SELECT expect — all four
# columns including total_exceptions, critical_exceptions
# and high_exceptions.
# --------------------------------------------------------
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, DoubleType
)

if portfolio_df.count() > 0:

    # Count accounts per source system
    source_accounts = portfolio_df.groupBy("source_system_code") \
        .agg(F.count("*").alias("total_accounts"))

    # Count exceptions per source system
    if exceptions_df.count() > 0:
        # Real exception data exists — aggregate it
        source_exceptions = exceptions_df.groupBy("source_system_code") \
            .agg(
                F.count("*").alias("total_exceptions"),
                F.sum(
                    F.when(F.col("severity_code") == "CRITICAL", 1)
                     .otherwise(0)
                ).alias("critical_exceptions"),
                F.sum(
                    F.when(F.col("severity_code") == "HIGH", 1)
                     .otherwise(0)
                ).alias("high_exceptions")
            )
    else:
        # No exceptions exist for this business date.
        # Create an empty DataFrame with the CORRECT schema so the
        # LEFT JOIN below works without column resolution errors.
        # This schema must match exactly what the SELECT references:
        #   source_system_code, total_exceptions,
        #   critical_exceptions, high_exceptions
        source_exceptions = spark.createDataFrame(
            [],
            StructType([
                StructField("source_system_code",  StringType(), True),
                StructField("total_exceptions",    LongType(),   True),
                StructField("critical_exceptions", LongType(),   True),
                StructField("high_exceptions",     LongType(),   True),
            ])
        )

    # LEFT JOIN: keep all source systems even if they have no exceptions.
    # COALESCE replaces NULL (no exceptions) with 0 for the calculation.
    source_metrics = source_accounts.alias("a").join(
        source_exceptions.alias("e"),
        F.col("a.source_system_code") == F.col("e.source_system_code"),
        "left"
        # left join: all sources in portfolio appear, even with 0 exceptions
    ).select(
        F.col("a.source_system_code"),
        F.col("a.total_accounts"),
        F.coalesce(F.col("e.total_exceptions"),    F.lit(0)).alias("total_exceptions"),
        F.coalesce(F.col("e.critical_exceptions"), F.lit(0)).alias("critical_exceptions"),
        F.coalesce(F.col("e.high_exceptions"),     F.lit(0)).alias("high_exceptions"),
        F.round(
            F.coalesce(F.col("e.total_exceptions"), F.lit(0)) * 100.0
            / F.col("a.total_accounts"),
            2
        ).alias("exception_rate")
        # exception_rate = what percentage of this source's accounts
        # have at least one exception
    )

    if debug:
        print("Source System KPIs:")
        source_metrics.orderBy(F.desc("exception_rate")).show(truncate=False)

else:
    # No portfolio data at all — create a fully empty source_metrics
    # with the complete schema so downstream cells do not fail
    source_metrics = spark.createDataFrame(
        [],
        StructType([
            StructField("source_system_code",  StringType(), True),
            StructField("total_accounts",      LongType(),   True),
            StructField("total_exceptions",    LongType(),   True),
            StructField("critical_exceptions", LongType(),   True),
            StructField("high_exceptions",     LongType(),   True),
            StructField("exception_rate",      DoubleType(), True),
        ])
    )

Source System KPIs:
+------------------+--------------+----------------+-------------------+---------------+--------------+
|source_system_code|total_accounts|total_exceptions|critical_exceptions|high_exceptions|exception_rate|
+------------------+--------------+----------------+-------------------+---------------+--------------+
|FOREX             |2             |2               |1                  |1              |100.0         |
|MOBILE            |2             |1               |0                  |1              |50.0          |
|LOANS             |4             |1               |1                  |0              |25.0          |
|CORE_BANKING      |4             |1               |1                  |0              |25.0          |
|CARDS             |4             |1               |0                  |1              |25.0          |
|INVESTMENTS       |3             |0               |0                  |0              |0.0           |
+------------------+--------------+---------

In [0]:
# Databricks notebook source
# ============================================================
# BUSINESS AREA KPIs
# 
# Each business area is measured by:
#   - Total exceptions assigned
#   - Critical exceptions assigned
#   - Escalated exceptions
#
# This helps identify which business areas have the most issues.
# ============================================================

if exceptions_df.count() > 0:
    business_area_kpis = exceptions_df.groupBy("business_area").agg(
        F.count("*").alias("total_exceptions"),
        F.sum(F.when(F.col("severity_code") == "CRITICAL", 1).otherwise(0)).alias("critical_exceptions"),
        F.sum(F.when(F.col("escalation_required") == True, 1).otherwise(0)).alias("escalated_exceptions")
    )
    
    if debug:
        print("Business Area KPIs:")
        business_area_kpis.orderBy(F.desc("total_exceptions")).show(truncate=False)
else:
    business_area_kpis = spark.createDataFrame([], 
        StructType([
            StructField("business_area", StringType(), True),
            StructField("total_exceptions", LongType(), True),
            StructField("critical_exceptions", LongType(), True),
            StructField("escalated_exceptions", LongType(), True)
        ])
    )

Business Area KPIs:
+-------------------+----------------+-------------------+--------------------+
|business_area      |total_exceptions|critical_exceptions|escalated_exceptions|
+-------------------+----------------+-------------------+--------------------+
|Risk               |3               |3                  |3                   |
|Customer Operations|3               |0                  |3                   |
+-------------------+----------------+-------------------+--------------------+



In [0]:
# Databricks notebook source
# ============================================================
# BUSINESS OWNER WORK QUEUE
# 
# Each business owner is measured by:
#   - Total exceptions assigned
#   - Critical exceptions assigned
#   - Earliest due date (most urgent)
#   - Latest due date (least urgent)
#
# This helps operations teams prioritise their work.
# ============================================================

if exceptions_df.count() > 0:
    owner_kpis = exceptions_df.groupBy("business_owner").agg(
        F.count("*").alias("assigned_exceptions"),
        F.sum(F.when(F.col("severity_code") == "CRITICAL", 1).otherwise(0)).alias("critical_exceptions"),
        F.min("resolution_due_date").alias("earliest_due_date"),
        F.max("resolution_due_date").alias("latest_due_date")
    )
    
    if debug:
        print("Business Owner KPIs:")
        owner_kpis.orderBy(F.desc("assigned_exceptions")).show(truncate=False)
else:
    owner_kpis = spark.createDataFrame([], 
        StructType([
            StructField("business_owner", StringType(), True),
            StructField("assigned_exceptions", LongType(), True),
            StructField("critical_exceptions", LongType(), True),
            StructField("earliest_due_date", TimestampType(), True),
            StructField("latest_due_date", TimestampType(), True)
        ])
    )

Business Owner KPIs:
+----------------------+-------------------+-------------------+-------------------+-------------------+
|business_owner        |assigned_exceptions|critical_exceptions|earliest_due_date  |latest_due_date    |
+----------------------+-------------------+-------------------+-------------------+-------------------+
|Risk Management       |3                  |3                  |2026-08-24 05:27:13|2026-08-24 05:27:13|
|Customer Services Team|3                  |0                  |2026-08-24 11:27:13|2026-08-24 11:27:13|
+----------------------+-------------------+-------------------+-------------------+-------------------+



In [0]:
# Databricks notebook source
# ============================================================
# BRANCH QUALITY METRICS
# 
# Each branch is measured by:
#   - Total accounts at that branch
#   - Total exceptions from that branch
#   - Branch exception rate
#
# This helps identify which branches have the most issues.
# ============================================================

if portfolio_df.count() > 0:
    # Get account counts per branch
    branch_accounts = portfolio_df.groupBy("branch_code") \
        .agg(F.count("*").alias("total_accounts"))
    
    # Get exception counts per branch
    # Join portfolio to exceptions by source_account_number
    if exceptions_df.count() > 0:
        # We need to join portfolio and exceptions to get branch-level exceptions
        branch_exceptions = portfolio_df.select(
            "source_account_number", "branch_code"
        ).alias("p").join(
            exceptions_df.select("source_account_number").alias("e"),
            F.col("p.source_account_number") == F.col("e.source_account_number"),
            "inner"
        ).groupBy("branch_code") \
         .agg(F.count("*").alias("total_exceptions"))
    else:
        branch_exceptions = spark.createDataFrame([], 
            StructType([
                StructField("branch_code", StringType(), True),
                StructField("total_exceptions", LongType(), True)
            ])
        )
    
    # Join to calculate exception rate
    branch_metrics = branch_accounts.alias("a").join(
        branch_exceptions.alias("e"),
        F.col("a.branch_code") == F.col("e.branch_code"),
        "left"
    ).select(
        F.col("a.branch_code"),
        F.col("a.total_accounts"),
        F.coalesce(F.col("e.total_exceptions"), F.lit(0)).alias("exceptions"),
        F.round(
            (F.coalesce(F.col("e.total_exceptions"), F.lit(0)) * 100.0) / F.col("a.total_accounts"), 2
        ).alias("branch_exception_rate")
    )
    
    if debug:
        print("Branch KPIs:")
        branch_metrics.orderBy(F.desc("branch_exception_rate")).show(truncate=False)
else:
    branch_metrics = spark.createDataFrame([], 
        StructType([
            StructField("branch_code", StringType(), True),
            StructField("total_accounts", LongType(), True),
            StructField("exceptions", LongType(), True),
            StructField("branch_exception_rate", DoubleType(), True)
        ])
    )

Branch KPIs:
+-----------+--------------+----------+---------------------+
|branch_code|total_accounts|exceptions|branch_exception_rate|
+-----------+--------------+----------+---------------------+
|BR001      |16            |6         |37.5                 |
|BR002      |3             |0         |0.0                  |
+-----------+--------------+----------+---------------------+



In [0]:
# Databricks notebook source
# ============================================================
# SLA PERFORMANCE
# 
# SLA Performance is measured by:
#   - Total exceptions by severity
#   - Overdue exceptions (ResolutionDueDate < current time)
#   - SLA Failure Rate (overdue / total * 100)
#
# This helps identify if the operations team is meeting SLAs.
# ============================================================

if exceptions_df.count() > 0:
    sla_kpis = exceptions_df.groupBy("severity_code").agg(
        F.count("*").alias("total_exceptions"),
        F.sum(F.when(F.col("resolution_due_date") < F.current_timestamp(), 1).otherwise(0)).alias("overdue_exceptions"),
        F.sum(F.when(F.col("escalation_required") == True, 1).otherwise(0)).alias("escalated_exceptions")
    ).withColumn(
        "sla_failure_rate",
        F.round(
            (F.col("overdue_exceptions") * 100.0) / F.col("total_exceptions"), 2
        )
    )
    
    if debug:
        print("SLA KPIs:")
        sla_kpis.orderBy(
            F.when(F.col("severity_code") == "CRITICAL", 1)
             .when(F.col("severity_code") == "HIGH", 2)
             .when(F.col("severity_code") == "MEDIUM", 3)
             .otherwise(4)
        ).show(truncate=False)
    
    # Calculate average SLA failure rate for compliance calculation
    avg_sla_failure = sla_kpis.select(F.avg("sla_failure_rate")).collect()[0][0] or 0
else:
    sla_kpis = spark.createDataFrame([], 
        StructType([
            StructField("severity_code", StringType(), True),
            StructField("total_exceptions", LongType(), True),
            StructField("overdue_exceptions", LongType(), True),
            StructField("escalated_exceptions", LongType(), True),
            StructField("sla_failure_rate", DoubleType(), True)
        ])
    )
    avg_sla_failure = 0

SLA KPIs:
+-------------+----------------+------------------+--------------------+----------------+
|severity_code|total_exceptions|overdue_exceptions|escalated_exceptions|sla_failure_rate|
+-------------+----------------+------------------+--------------------+----------------+
|CRITICAL     |3               |0                 |3                   |0.0             |
|HIGH         |3               |0                 |3                   |0.0             |
+-------------+----------------+------------------+--------------------+----------------+



In [0]:
# Databricks notebook source
# ============================================================
# 30-DAY TREND ANALYSIS
# 
# The trend analysis shows how exceptions have changed
# over the last 30 days.
#
# This helps identify if data quality is improving or worsening.
# ============================================================

# Calculate the date 30 days ago
from datetime import timedelta
date_30_days_ago = (datetime.strptime(business_date, "%Y-%m-%d") - timedelta(days=30)).strftime("%Y-%m-%d")

trend_df = spark.table("retailbank_dev.warehouse.customer_portfolio_exceptions") \
    .filter(
        (F.col("business_date") >= date_30_days_ago) & 
        (F.col("business_date") <= business_date)
    ) \
    .groupBy("business_date").agg(
        F.count("*").alias("total_exceptions"),
        F.sum(F.when(F.col("severity_code") == "CRITICAL", 1).otherwise(0)).alias("critical_exceptions"),
        F.countDistinct("business_owner").alias("active_business_owners")
    ) \
    .orderBy("business_date")

if debug:
    print(f"30-Day Trend Analysis ({date_30_days_ago} to {business_date}):")
    if trend_df.count() > 0:
        trend_df.show(20, truncate=False)
    else:
        print("  No data found for trend analysis.")

30-Day Trend Analysis (2026-01-01 to 2026-01-31):
+-------------+----------------+-------------------+----------------------+
|business_date|total_exceptions|critical_exceptions|active_business_owners|
+-------------+----------------+-------------------+----------------------+
|2026-01-31   |6               |3                  |2                     |
+-------------+----------------+-------------------+----------------------+



In [0]:
# Databricks notebook source
# ============================================================
# EXECUTIVE SCORECARD
# 
# The Executive Scorecard consolidates the 4 key KPIs
# with GREEN/AMBER/RED status indicators.
#
#   PORTFOLIO HEALTH:    EXCELLENT/GOOD = GREEN, FAIR = AMBER, POOR = RED
#   CRITICAL EXCEPTIONS: < 10 = GREEN, 10-50 = AMBER, > 50 = RED
#   EXCEPTION RATE:      < 2% = GREEN, 2-5% = AMBER, > 5% = RED
#   SLA COMPLIANCE:      > 95% = GREEN, 85-95% = AMBER, < 85% = RED
# ============================================================

from pyspark.sql.types import StructType, StructField, StringType

# Define schema for scorecard
scorecard_schema = StructType([
    StructField("kpi_name", StringType(), True),
    StructField("kpi_value", StringType(), True),
    StructField("kpi_status", StringType(), True)
])

# Build scorecard rows
scorecard_rows = []

# 1. Portfolio Health
health_status = "GREEN" if portfolio_health in ("EXCELLENT", "GOOD") else "AMBER" if portfolio_health == "FAIR" else "RED"
scorecard_rows.append(("Portfolio Health", portfolio_health, health_status))

# 2. Critical Exceptions
crit_status = "GREEN" if critical_exceptions < 10 else "AMBER" if critical_exceptions < 50 else "RED"
scorecard_rows.append(("Critical Exceptions", str(critical_exceptions), crit_status))

# 3. Exception Rate
rate_status = "GREEN" if exception_rate < 2 else "AMBER" if exception_rate < 5 else "RED"
scorecard_rows.append(("Exception Rate", f"{exception_rate}%", rate_status))

# 4. SLA Compliance
sla_compliance = round(100 - avg_sla_failure, 2)
sla_status = "GREEN" if sla_compliance > 95 else "AMBER" if sla_compliance > 85 else "RED"
scorecard_rows.append(("SLA Compliance", f"{sla_compliance}%", sla_status))

# Create DataFrame
scorecard_df = spark.createDataFrame(scorecard_rows, scorecard_schema)

if debug:
    print("Executive Scorecard:")
    scorecard_df.show(truncate=False)

Executive Scorecard:
+-------------------+---------+----------+
|kpi_name           |kpi_value|kpi_status|
+-------------------+---------+----------+
|Portfolio Health   |POOR     |RED       |
|Critical Exceptions|3        |GREEN     |
|Exception Rate     |31.58%   |RED       |
|SLA Compliance     |100%     |GREEN     |
+-------------------+---------+----------+



In [0]:
# Databricks notebook source
# ============================================================
# LOAD REPORTING.PORTFOLIO_EXCEPTION_REPORT
# 
# The reporting table is truncated and reloaded with the
# current day's scorecard. This provides a simple view for
# executive dashboards.
#
# This matches the SQL behavior: TRUNCATE then INSERT.
# ============================================================

target_table = "retailbank_dev.reporting.portfolio_exception_report"

# Truncate the table
spark.sql(f"TRUNCATE TABLE {target_table}")

# Insert scorecard data
scorecard_to_insert = scorecard_df.select(
    F.lit(business_date).cast("date").alias("business_date"),
    F.col("kpi_name"),
    F.col("kpi_value"),
    F.col("kpi_status"),
    F.current_timestamp().alias("created_date")
)

scorecard_to_insert.write.format("delta").mode("append").saveAsTable(target_table)

rows_inserted = scorecard_to_insert.count()

if debug:
    print(f"Reporting table loaded: {rows_inserted} rows inserted into {target_table}")
    spark.sql(f"SELECT * FROM {target_table} WHERE business_date = '{business_date}'").show(truncate=False)

Reporting table loaded: 4 rows inserted into retailbank_dev.reporting.portfolio_exception_report
+---------+-------------+-------------------+---------+----------+--------------------------+
|report_id|business_date|kpi_name           |kpi_value|kpi_status|created_date              |
+---------+-------------+-------------------+---------+----------+--------------------------+
|101      |2026-01-31   |Portfolio Health   |POOR     |RED       |2026-08-24 03:28:11.705526|
|102      |2026-01-31   |Critical Exceptions|3        |GREEN     |2026-08-24 03:28:11.705526|
|103      |2026-01-31   |Exception Rate     |31.58%   |RED       |2026-08-24 03:28:11.705526|
|104      |2026-01-31   |SLA Compliance     |100%     |GREEN     |2026-08-24 03:28:11.705526|
+---------+-------------+-------------------+---------+----------+--------------------------+



In [0]:
# Databricks notebook source
# ============================================================
# AUDIT: REPORTING EXECUTION SUMMARY
# 
# Tracks which reports were generated and how long they took.
# ============================================================

end_time         = datetime.now()
duration_seconds = int((end_time - start_time).total_seconds())

spark.sql(f"""
    INSERT INTO retailbank_dev.audit.reporting_execution_summary
    (business_date, execution_id, procedure_name, report_name, records_generated, execution_seconds, created_date)
    VALUES
    ('{business_date}', '{execution_id}', '{procedure_name}', 
     'Portfolio Exception Dashboard', {rows_inserted}, 
     {duration_seconds}, '{end_time.strftime("%Y-%m-%d %H:%M:%S")}')
""")

if debug:
    print("Reporting execution summary written.")

Reporting execution summary written.


In [0]:
# Databricks notebook source
# ============================================================
# CELL 15: AUDIT - EXECUTIVE DASHBOARD METRICS
# 
# Snapshot of portfolio health for the executive dashboard.
# ============================================================

spark.sql(f"""
    INSERT INTO retailbank_dev.audit.executive_dashboard_metrics
    (business_date, portfolio_health, critical_exceptions, exception_rate, sla_compliance, generated_date)
    VALUES
    ('{business_date}', '{portfolio_health}', {critical_exceptions}, {exception_rate}, {sla_compliance}, '{end_time.strftime("%Y-%m-%d %H:%M:%S")}')
""")

if debug:
    print("Executive dashboard metrics written.")

Executive dashboard metrics written.


In [0]:
# Databricks notebook source
# ============================================================
# CELL 16: FINAL AUDIT UPDATE
# 
# Updates the ETL execution log with final status and statistics.
# ============================================================

status  = "SUCCESS"
message = (
    f"Portfolio Exception Report generated successfully. "
    f"Report Records: {rows_inserted}, "
    f"Execution Time: {duration_seconds} seconds."
)

write_audit_end(status, message)

if debug:
    print("Final audit log updated.")

Final audit log updated.


In [0]:
# Databricks notebook source
# ============================================================
# CELL 17: SUMMARY OUTPUT
# 
# Displays the final summary of the portfolio exception report.
# This matches the output style of nb_03 and nb_04.
# ============================================================

if debug:
    print("=" * 50)
    print("PORTFOLIO EXCEPTION REPORT SUMMARY")
    print("=" * 50)
    print(f"Business Date       : {business_date}")
    print(f"Total Customers     : {total_customers}")
    print(f"Total Accounts      : {total_accounts}")
    print(f"Total Exceptions    : {total_exceptions}")
    print(f"Exception Rate      : {exception_rate}%")
    print(f"Portfolio Health    : {portfolio_health}")
    print(f"Executive Status    : {executive_status}")
    print(f"SLA Compliance      : {sla_compliance}%")
    print(f"Report Records      : {rows_inserted}")
    print(f"Execution Time      : {duration_seconds} seconds")
    print(f"Status              : {status}")
    print("=" * 50)
    
    print("\nExecutive Scorecard:")
    scorecard_df.show(truncate=False)
    
    print("\nSource System KPIs:")
    if source_metrics.count() > 0:
        source_metrics.orderBy(F.desc("exception_rate")).show(truncate=False)
    
    print("\nSLA KPIs:")
    if sla_kpis.count() > 0:
        sla_kpis.orderBy("severity_code").show(truncate=False)

print("nb_05_PortfolioExceptionReport completed successfully.")

PORTFOLIO EXCEPTION REPORT SUMMARY
Business Date       : 2026-01-31
Total Customers     : 12
Total Accounts      : 19
Total Exceptions    : 6
Exception Rate      : 31.58%
Portfolio Health    : POOR
Executive Status    : Normal Operations
SLA Compliance      : 100%
Report Records      : 4
Execution Time      : 19 seconds
Status              : SUCCESS

Executive Scorecard:
+-------------------+---------+----------+
|kpi_name           |kpi_value|kpi_status|
+-------------------+---------+----------+
|Portfolio Health   |POOR     |RED       |
|Critical Exceptions|3        |GREEN     |
|Exception Rate     |31.58%   |RED       |
|SLA Compliance     |100%     |GREEN     |
+-------------------+---------+----------+


Source System KPIs:
+------------------+--------------+----------------+-------------------+---------------+--------------+
|source_system_code|total_accounts|total_exceptions|critical_exceptions|high_exceptions|exception_rate|
+------------------+--------------+----------------+